In [1]:
"""
compare_moments.py
==================
Reads the simulated moment CSVs (both Fortran and Python versions) and the target .dat files,
then produces one PDF per moment type with comparison plots showing all three.

Usage:
    python compare_moments.py

Outputs:  fig_1_SdSkewKurt_L1.pdf
          fig_2_SdSkewKurt_L5.pdf
          fig_3_impulse_response.pdf
          fig_4_lifetime_income_growth.pdf
          fig_5_var_lny.pdf
          fig_6_EmpCDF.pdf
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import os

# ── Macro ──────────────────────────────────────────────────────────
fp_data = r'C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\data'
fp_outp = r'C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output'

# ── colour palette ──────────────────────────────────────────────────────────
FORT_COL = "#2166ac"   # blue    – Fortran simulated
PY_COL   = "#1b9e77"   # green   – Python simulated
DAT_COL  = "#d6604d"   # red     – data target

plt.rcParams.update({
    "font.family":      "serif",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.linestyle":   "--",
    "figure.dpi":       120,
})

LEGEND_HANDLES = [
    Line2D([0],[0], color=FORT_COL, lw=2, marker='o', ms=5, label='Fortran sim'),
    Line2D([0],[0], color=PY_COL,   lw=2, marker='^', ms=5, label='Python sim'),
    Line2D([0],[0], color=DAT_COL,  lw=2, marker='s', ms=5, ls='--', label='Data target'),
]

# ── bin labels ───────────────────────────────────────────────────────────────
VASE_BINS   = ["P1-2","P2-11","P11-21","P21-31","P31-41","P41-51",
               "P51-61","P61-71","P71-81","P81-91","P91-96","P96-100","Top 1%"]
IR_INC_BINS = ["P1-6","P6-11","P11-31","P31-51","P51-71","P71-91","P91-96","P96-100"]
IR_SHK_BINS = ["P1-3","P3-6","P6-11","P11-31","P31-51",
               "P51-71","P71-91","P91-96","P96-99","Top 1%"]
LT_BINS     = ["P1-2","P2-6","P6-11","P11-21","P21-31","P31-41","P41-51",
               "P51-61","P61-71","P71-81","P81-91","P91-96","P96-98","P98-100","Top 1%"]
LAG_LABELS  = ["Impact","Lag 1yr","Lag 2yr","Lag 3yr","Lag 5yr","Lag 10yr"]
AGE_LABELS_CS = ["Young\n(~27–34)","Middle\n(~33–47)","Old\n(~43–58)"]
AGE_LABELS_IR = ["Young\n(~27–34)","Old\n(~35–50)"]
PERIOD_AGES = [25, 30, 35, 40, 45, 50, 55, 60]

# ============================================================================
# LOAD DATA
# ============================================================================

def load_ssk_sim(path):
    """Load SdSkewKurt sim CSV -> dict[age_group][moment] = array(13,)"""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df['moment']    = df['moment'].str.strip()
    df['age_group'] = df['age_group'].str.strip()
    out = {}
    for ag in ['YNG','MID','OLD']:
        out[ag] = {}
        for mom in ['sd','skew','kurt']:
            mask = (df['age_group']==ag) & (df['moment']==mom)
            out[ag][mom] = df.loc[mask,'value'].values
    return out

def load_ssk_dat(path):
    """Load SdSkewKurt .dat -> dict[age_group][moment] = array(13,)
       Fortran layout: 39 rows x 3 cols, rows 1-13=YNG, 14-26=MID, 27-39=OLD
       cols: sd, skew, kurt
    """
    raw = np.loadtxt(path)   # (39,3)
    names = ['YNG','MID','OLD']
    moms  = ['sd','skew','kurt']
    out = {}
    for g, name in enumerate(names):
        out[name] = {}
        block = raw[g*13:(g+1)*13, :]
        for m, mom in enumerate(moms):
            out[name][mom] = block[:, m]
    return out

def load_ir_sim(path):
    """Load irmoments sim CSV -> array(2,8,10,6)"""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    arr = np.zeros((2,8,10,6))
    for _, row in df.iterrows():
        a = int(row['age_group'])-1
        i = int(row['income_bin'])-1
        k = int(row['shock_bin'])-1
        l = int(row['lag'])-1
        arr[a,i,k,l] = row['value']
    return arr

def load_ir_dat(path):
    """Load ImpulseA_mean.dat -> array(2,8,23,6)
       Fortran layout: (2*8*23) rows x 6 cols
       Dirmoments(h,i,j,:) = row[(h-1)*8*23+(i-1)*23+j]
    """
    raw = np.loadtxt(path)   # (368,6)
    arr = np.zeros((2,8,23,6))
    for h in range(2):
        for i in range(8):
            for j in range(23):
                row = h*8*23 + i*23 + j
                arr[h,i,j,:] = raw[row,:]
    return arr

def load_incgrwth_sim(path):
    """Load incgrwth sim CSV -> array(15,8)"""
    raw = pd.read_csv(path, header=None).values   # (120,3)
    arr = np.zeros((15,8))
    for row in raw:
        i = int(row[0])-1
        j = int(row[1])-1
        arr[i,j] = row[2]
    return arr

def load_incgrwth_dat(path):
    """Load meanLTinc_level.dat -> array(15,8)"""
    return np.loadtxt(path)   # (15,8)

def load_varlny_sim(path):
    raw = pd.read_csv(path, header=None).values   # (36,2)
    return raw[:,1]

def load_varlny_dat(path):
    return np.loadtxt(path)   # (36,)

def load_empcdf_sim(path):
    raw = pd.read_csv(path, header=None).values   # (37,2)
    return raw[:,1]

def load_empcdf_dat(path):
    return np.loadtxt(path)   # (37,)

# ============================================================================
# FIGURE 1 & 2:  SdSkewKurt  (L1 and L5)
# ============================================================================

def plot_ssk(sim_fort, sim_py, dat, title, filename):
    """
    3 rows (age groups) x 3 cols (sd / skew / kurt)
    Each panel: x-axis = income bin 1..13, three lines (Fortran, Python, Data)
    """
    age_groups = ['YNG','MID','OLD']
    moments    = ['sd','skew','kurt']
    mom_labels = ['Std Dev of income change',
                  'Skewness of income change',
                  'Excess Kurtosis of income change']
    x = np.arange(1,14)

    fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharey='col')
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    for row, ag in enumerate(age_groups):
        for col, (mom, mlabel) in enumerate(zip(moments, mom_labels)):
            ax = axes[row, col]

            sf = sim_fort[ag][mom]
            sp = sim_py[ag][mom]
            d  = dat[ag][mom]

            ax.plot(x, sf, color=FORT_COL, lw=2, marker='o', ms=4, label='Fortran')
            ax.plot(x, sp, color=PY_COL,   lw=2, marker='^', ms=4, label='Python')
            ax.plot(x, d,  color=DAT_COL,  lw=2, marker='s', ms=4, ls='--', label='Data')
            ax.axhline(0, color='k', lw=0.5, alpha=0.4)

            if row == 0:
                ax.set_title(mlabel, fontsize=10, fontweight='bold')
            if col == 0:
                ax.set_ylabel(AGE_LABELS_CS[row], fontsize=9, rotation=0,
                              ha='right', labelpad=50)
            if row == 2:
                ax.set_xlabel('Income bin (low → high)', fontsize=8)
                ax.set_xticks(x[::2])
                ax.set_xticklabels(VASE_BINS[::2], rotation=45, ha='right', fontsize=6)
            else:
                ax.set_xticks([])

    fig.legend(handles=LEGEND_HANDLES, loc='upper center',
               ncol=3, bbox_to_anchor=(0.5, 1.0), fontsize=10,
               frameon=True, framealpha=0.9)
    fig.tight_layout()
    fig.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved {filename}")

# ============================================================================
# FIGURE 3:  Impulse Responses
# ============================================================================

def ir_dat_to_10bins(dat_arr):
    """
    Interpolate the 23-bin data IRF to the 10-bin sim grid.
    """
    sim_bounds  = np.array([1,3,6,11,31,51,71,91,96,99,101])
    sim_mids    = (sim_bounds[:-1] + sim_bounds[1:]) / 2.0  # 10 values
    
    dat_bounds2 = np.array([1,3,6,11,16,21,26,31,36,41,46,51,56,61,66,71,76,81,86,91,96,99,101,101])
    dat_mids2   = (dat_bounds2[:-1] + dat_bounds2[1:]) / 2.0
    
    out = np.zeros((2,8,10,6))
    for h in range(2):
        for i in range(8):
            for k, sm in enumerate(sim_mids):
                nearest = np.argmin(np.abs(dat_mids2 - sm))
                out[h,i,k,:] = dat_arr[h,i,nearest,:]
    return out

def plot_impulse(ir_fort, ir_py, ir_dat_raw, filename):
    """
    2 age groups x 8 income bins.
    Each panel: x = lag (0..5), lines per shock bin.
    Three line styles: Fortran (solid circles), Python (solid triangles), Data (dashed squares)
    """
    ir_dat = ir_dat_to_10bins(ir_dat_raw)

    # select 5 representative shock bins
    shock_sel   = [0, 2, 4, 7, 9]   # 0-indexed: bins 1,3,5,8,10
    shock_names = ["Shock bin 1\n(most negative)",
                   "Shock bin 3",
                   "Shock bin 5\n(near zero)",
                   "Shock bin 8",
                   "Shock bin 10\n(most positive)"]
    cmap = plt.cm.RdYlBu
    shock_cols = [cmap(v) for v in [0.05, 0.25, 0.5, 0.75, 0.95]]

    x = np.arange(6)

    for age_idx, age_label in enumerate(AGE_LABELS_IR):
        fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=True, sharex=True)
        fig.suptitle(
            f"Impulse Response Moments — {age_label.replace(chr(10),' ')}\n"
            f"(Each panel = income bin, colored lines = shock bins, line style = source)",
            fontsize=12, fontweight='bold'
        )

        for inc_idx in range(8):
            row = inc_idx // 4
            col = inc_idx %  4
            ax  = axes[row, col]

            for s_i, (si, sname, scol) in enumerate(zip(shock_sel, shock_names, shock_cols)):
                fort_line = ir_fort[age_idx, inc_idx, si, :]
                py_line   = ir_py[age_idx, inc_idx, si, :]
                dat_line  = ir_dat[age_idx, inc_idx, si, :]
                
                ax.plot(x, fort_line, color=scol, lw=2, marker='o', ms=3, alpha=0.8)
                ax.plot(x, py_line,   color=scol, lw=2, marker='^', ms=3, alpha=0.8)
                ax.plot(x, dat_line,  color=scol, lw=2, marker='s', ms=3, ls='--', alpha=0.8)

            ax.axhline(0, color='k', lw=0.6, alpha=0.4)
            ax.set_title(f"Income bin {inc_idx+1}\n({IR_INC_BINS[inc_idx]})",
                         fontsize=8, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(LAG_LABELS, rotation=45, ha='right', fontsize=6)
            if col == 0:
                ax.set_ylabel('Mean arc % change', fontsize=8)

        # shock-bin colour legend
        shock_handles = [
            Line2D([0],[0], color=shock_cols[k], lw=2, label=shock_names[k])
            for k in range(len(shock_sel))
        ]
        style_handles = [
            Line2D([0],[0], color='gray', lw=2, marker='o', label='Fortran sim'),
            Line2D([0],[0], color='gray', lw=2, marker='^', label='Python sim'),
            Line2D([0],[0], color='gray', lw=2, marker='s', ls='--', label='Data target'),
        ]
        fig.legend(handles=shock_handles + style_handles,
                   loc='lower center', ncol=4,
                   bbox_to_anchor=(0.5, -0.05), fontsize=7, frameon=True)
        fig.tight_layout()
        fname = filename.replace('.pdf', f'_age{age_idx+1}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved {fname}")

# ============================================================================
# FIGURE 4:  Lifetime income growth
# ============================================================================

def plot_incgrwth(sim_fort, sim_py, dat, filename):
    """
    15 panels (one per lifetime income bin), x=period (age), three lines.
    Arranged 3 rows x 5 cols.
    """
    fig, axes = plt.subplots(3, 5, figsize=(16, 9),
                             sharey=False, sharex=True)
    fig.suptitle(
        "Lifetime Income Growth by Lifetime Income Percentile\n"
        "(x-axis = age snapshot, y-axis = mean income $000s)",
        fontsize=13, fontweight='bold'
    )

    for idx in range(15):
        row = idx // 5
        col = idx %  5
        ax  = axes[row, col]

        ax.plot(PERIOD_AGES, sim_fort[idx,:], color=FORT_COL, lw=2,
                marker='o', ms=4, label='Fortran')
        ax.plot(PERIOD_AGES, sim_py[idx,:],   color=PY_COL, lw=2,
                marker='^', ms=4, label='Python')
        ax.plot(PERIOD_AGES, dat[idx,:],      color=DAT_COL, lw=2,
                marker='s', ms=4, ls='--', label='Data')

        ax.set_title(f"Bin {idx+1}: {LT_BINS[idx]}", fontsize=8, fontweight='bold')
        if col == 0:
            ax.set_ylabel('Mean income ($000s)', fontsize=7)
        if row == 2:
            ax.set_xlabel('Age', fontsize=7)
            ax.set_xticks(PERIOD_AGES[::2])

    fig.legend(handles=LEGEND_HANDLES, loc='upper center',
               ncol=3, bbox_to_anchor=(0.5, 1.01), fontsize=10,
               frameon=True, framealpha=0.9)
    fig.tight_layout()
    fig.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved {filename}")

# ============================================================================
# FIGURE 5:  Variance of log income
# ============================================================================

def plot_varlny(sim_fort, sim_py, dat, filename):
    ages = np.arange(25, 61)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # left: levels
    ax = axes[0]
    ax.plot(ages, sim_fort, color=FORT_COL, lw=2, marker='o', ms=4, label='Fortran')
    ax.plot(ages, sim_py,   color=PY_COL,   lw=2, marker='^', ms=4, label='Python')
    ax.plot(ages, dat,      color=DAT_COL,  lw=2, marker='s', ms=4, ls='--', label='Data')
    ax.set_title('Variance of Log Income by Age', fontsize=11, fontweight='bold')
    ax.set_xlabel('Age'); ax.set_ylabel('Var(log income)')
    ax.legend(fontsize=9)

    # right: differences from data
    ax = axes[1]
    diff_fort = sim_fort - dat
    diff_py   = sim_py - dat
    
    width = 0.4
    ax.bar(ages - width/2, diff_fort, width, color=FORT_COL, alpha=0.7, label='Fortran - Data')
    ax.bar(ages + width/2, diff_py,   width, color=PY_COL,   alpha=0.7, label='Python - Data')
    ax.axhline(0, color='k', lw=1)
    ax.set_title('Differences from Data', fontsize=11, fontweight='bold')
    ax.set_xlabel('Age'); ax.set_ylabel('Difference')
    ax.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved {filename}")

# ============================================================================
# FIGURE 6:  Employment CDF
# ============================================================================

def plot_empcdf(sim_fort, sim_py, dat, filename):
    x = np.arange(0, 37)   # 0..36 employment years
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # left: CDF curves
    ax = axes[0]
    ax.plot(x, sim_fort, color=FORT_COL, lw=2, marker='o', ms=4, label='Fortran')
    ax.plot(x, sim_py,   color=PY_COL,   lw=2, marker='^', ms=4, label='Python')
    ax.plot(x, dat,      color=DAT_COL,  lw=2, marker='s', ms=4, ls='--', label='Data')
    ax.set_title('Cumulative Distribution of Employment Years', fontsize=11, fontweight='bold')
    ax.set_xlabel('Number of employed years (out of 36)')
    ax.set_ylabel('Cumulative % of workers')
    ax.legend(fontsize=9)

    # right: differences from data
    ax = axes[1]
    diff_fort = sim_fort - dat
    diff_py   = sim_py - dat
    
    width = 0.4
    ax.bar(x - width/2, diff_fort, width, color=FORT_COL, alpha=0.7, label='Fortran - Data')
    ax.bar(x + width/2, diff_py,   width, color=PY_COL,   alpha=0.7, label='Python - Data')
    ax.axhline(0, color='k', lw=1)
    ax.set_title('Differences from Data', fontsize=11, fontweight='bold')
    ax.set_xlabel('Number of employed years (out of 36)')
    ax.set_ylabel('Difference (pp)')
    ax.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(filename, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved {filename}")

# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    os.makedirs(rf'{fp_outp}\figures', exist_ok=True)

    print("Loading data ...")

    # Load Fortran-generated moments (no _toolbox suffix)
    ssk_l1_fort = load_ssk_sim(rf'{fp_data}\intermediate\SdSkewKurt_L1_sim.csv')
    ssk_l5_fort = load_ssk_sim(rf'{fp_data}\intermediate\SdSkewKurt_L5_sim.csv')
    ir_fort     = load_ir_sim(rf'{fp_data}\intermediate\irmoments_sim.csv')
    lg_fort     = load_incgrwth_sim(rf'{fp_data}\intermediate\incgrwth_sim.csv')
    vl_fort     = load_varlny_sim(rf'{fp_data}\intermediate\var_lny_sim.csv')
    ec_fort     = load_empcdf_sim(rf'{fp_data}\intermediate\EmpCDF_sim.csv')

    # Load Python-generated moments (_toolbox suffix)
    ssk_l1_py = load_ssk_sim(rf'{fp_data}\intermediate\SdSkewKurt_L1_sim_toolbox.csv')
    ssk_l5_py = load_ssk_sim(rf'{fp_data}\intermediate\SdSkewKurt_L5_sim_toolbox.csv')
    ir_py     = load_ir_sim(rf'{fp_data}\intermediate\irmoments_sim_toolbox.csv')
    lg_py     = load_incgrwth_sim(rf'{fp_data}\intermediate\incgrwth_sim_toolbox.csv')
    vl_py     = load_varlny_sim(rf'{fp_data}\intermediate\var_lny_sim_toolbox.csv')
    ec_py     = load_empcdf_sim(rf'{fp_data}\intermediate\EmpCDF_sim_toolbox.csv')

    # Load target data
    ssk_l1_dat = load_ssk_dat(rf'{fp_data}\intermediate\SdSkewKurt_L1.dat')
    ssk_l5_dat = load_ssk_dat(rf'{fp_data}\intermediate\SdSkewKurt_L5.dat')
    ir_dat_raw = load_ir_dat(rf'{fp_data}\intermediate\ImpulseA_mean.dat')
    lg_dat     = load_incgrwth_dat(rf'{fp_data}\intermediate\meanLTinc_level.dat')
    vl_dat     = load_varlny_dat(rf'{fp_data}\intermediate\var_lny.dat')
    ec_dat     = load_empcdf_dat(rf'{fp_data}\intermediate\EmpCDF.dat')

    print("\nGenerating figures ...")

    plot_ssk(ssk_l1_fort, ssk_l1_py, ssk_l1_dat,
             "Cross-Sectional Moments: 1-Year Income Changes (Std Dev / Skewness / Kurtosis)",
             rf'{fp_outp}\figures\fig_1_SdSkewKurt_L1.pdf')

    plot_ssk(ssk_l5_fort, ssk_l5_py, ssk_l5_dat,
             "Cross-Sectional Moments: 5-Year Income Changes (Std Dev / Skewness / Kurtosis)",
             rf'{fp_outp}\figures\fig_2_SdSkewKurt_L5.pdf')

    plot_impulse(ir_fort, ir_py, ir_dat_raw,
                 rf'{fp_outp}\figures\fig_3_impulse_response.pdf')

    plot_incgrwth(lg_fort, lg_py, lg_dat,
                  rf'{fp_outp}\figures\fig_4_lifetime_income_growth.pdf')

    plot_varlny(vl_fort, vl_py, vl_dat,
                rf'{fp_outp}\figures\fig_5_var_lny.pdf')

    plot_empcdf(ec_fort, ec_py, ec_dat,
                rf'{fp_outp}\figures\fig_6_EmpCDF.pdf')

    print(rf"\nDone. All figures written to {fp_outp}\figures")

Loading data ...

Generating figures ...
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_1_SdSkewKurt_L1.pdf
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_2_SdSkewKurt_L5.pdf
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_3_impulse_response_age1.pdf
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_3_impulse_response_age2.pdf
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_4_lifetime_income_growth.pdf
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_5_var_lny.pdf
  Saved C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures\fig_6_EmpCDF.pdf
\nDone. All figures written to C:\Users\adv23\Dropbox\John-Jackson-Steve 2023\earning_dynamics\output\figures
